# 18.14 蒸餾後再量化能省多少？

# 第 18 章：蒸餾，讓小模型向教師學習

前置：CE、模型輸出與學生訓練。教師像經驗較多的老師：可以只給答案，也可展示對各候選的分配。老師可能說錯；學生也須真的選較小架構。此處只驗loss與梯度，正式蒸餾需讀者提供已訓練教師。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：老師不只說第一名，也指出接近的候選**

學生選較小架構；老師可能教錯，仍要真值與行爲評估。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/distillation.svg")))

**先想一想：**只報teacher與量化學生，能知道蒸餾有沒有幫助嗎？

先由更小學生學習，再把學生的數值存得更緊，是兩條省成本路線組合。要保留未蒸餾學生，才知道品質來自哪項。

**把直覺寫成數學：**四組：teacher、SFT student、distilled student、quantized distilled student；成本與品質並排。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig
from tiny_perceptron.quantization import replace_linear_layers

student = TinyLM(ModelConfig(width=8))
before = sum(p.numel() * p.element_size() for p in student.parameters())
compressed = replace_linear_layers(student, 4)
after = sum(t.numel() * t.element_size() for t in list(compressed.parameters()) + list(compressed.buffers()))
print("隨機學生轉換payload", before, after, "品質仍需四組真實checkpoint評估")

**如何讀結果：**不把隨機學生轉換叫「蒸餾完成」；真的組合要先執行蒸餾，再PTQ與行爲retention測試。

**只改一件事：**只選擇8bit，記錄儲存與誤差取捨。
